# Week 3: Reasoning and Tool-Use Foundations
## ReAct: Interleaved Reasoning and Action

### Overview
This notebook implements a minimal single-tool ReAct-style loop inspired by **ReAct: Synergizing Reasoning and Acting in Language Models** (Yao et al., 2022). The experiment compares direct answering with a ReAct-style agent that can interleave reasoning, a calculator action, and tool observations.

## 1. Overview & Objective

**Fellowship task:** Hand-implement a minimal single-tool ReAct-style loop on a small task.

### Objective
Test whether explicitly giving a language model access to a tool and allowing it to alternate between reasoning and action can improve performance on simple arithmetic tasks. This is a small demonstration of the mechanism, not a reproduction of the paper's full experiments.

## 2. Setup & Task Environment

We use an open-source instruction-tuned model and one deterministic calculator tool. The same fixed questions are given to both conditions.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 118.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


In [ ]:
import pandas as pd
import torch
from week3_react import MODEL_NAME, load_model, safe_calculate, baseline_answer, react_answer, run_experiment
print("Model:", MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

Model: Qwen/Qwen2.5-3B-Instruct
CUDA available: True


In [ ]:
TASKS = [
    {'id': 'Q01', 'question': 'A jacket costs £80. It is discounted by 25%, and then a further £6 is added for delivery. What is the final amount paid?', 'answer': '66'},
    {'id': 'Q02', 'question': 'A student has £150. They spend 35% of it on books and then spend another £24. How much money remains?', 'answer': '73.5'},
    {'id': 'Q03', 'question': 'A train travels 125 km in the morning and 18% farther in the afternoon. How many kilometres does it travel altogether?', 'answer': '272.5'},
    {'id': 'Q04', 'question': 'A company has 240 employees. 15% work remotely and 25% of the remaining employees work part-time. How many full-time employees work on-site?', 'answer': '153'},
    {'id': 'Q05', 'question': 'A phone costs £600. It is reduced by 20%, then the discounted price is reduced by another 10%. What is the final price?', 'answer': '432'},
    {'id': 'Q06', 'question': 'A restaurant bill is £84. A service charge of 12.5% is added. What is the total bill?', 'answer': '94.5'},
    {'id': 'Q07', 'question': 'A machine produces 175 units per hour for 6 hours. In the final hour it produces 12% fewer units than in each of the earlier hours. How many units are produced in total?', 'answer': '1029'},
    {'id': 'Q08', 'question': 'A class has 48 students. Two-thirds submit an assignment on time. Of those who submit on time, 75% receive a distinction. How many students receive a distinction?', 'answer': '24'},
    {'id': 'Q09', 'question': 'A shop buys 80 items at £12 each and sells each item for £18. What is the total profit?', 'answer': '480'},
    {'id': 'Q10', 'question': 'A monthly electricity bill is £120. It increases by 15% and then receives a £10 discount. What is the final bill?', 'answer': '128'},
    {'id': 'Q11', 'question': 'A car travels 360 miles using 12 gallons of fuel. At the same efficiency, how many gallons are needed to travel 525 miles?', 'answer': '17.5'},
    {'id': 'Q12', 'question': 'A project has 480 tasks. 35% are completed in week one and 25% of the remaining tasks are completed in week two. How many tasks remain?', 'answer': '234'},
    {'id': 'Q13', 'question': 'A worker earns £16.50 per hour for the first 35 hours and 1.5 times that rate for each of 6 overtime hours. What is the total weekly pay?', 'answer': '726'},
    {'id': 'Q14', 'question': 'A recipe serves 6 people and requires 450 grams of rice. How much rice is needed for 14 people?', 'answer': '1050'},
    {'id': 'Q15', 'question': 'A product originally costs £240. The price is increased by 18% and then reduced by 15%. What is the final price?', 'answer': '240.72'},
    {'id': 'Q16', 'question': 'A university has 1,200 students. 40% study computing, and 30% of computing students are postgraduate students. How many postgraduate computing students are there?', 'answer': '144'},
    {'id': 'Q17', 'question': 'A warehouse receives 3 shipments containing 125, 180, and 245 items. It then ships out 28% of the total. How many items remain?', 'answer': '396'},
    {'id': 'Q18', 'question': 'An investment of £800 increases by 12% in the first year and decreases by 5% in the second year. What is its value after two years?', 'answer': '851.2'},
    {'id': 'Q19', 'question': 'A team completes 45% of a 320-task project in the first phase. In the second phase, it completes 40% of the remaining tasks. How many tasks are still unfinished?', 'answer': '105.6'},
    {'id': 'Q20', 'question': 'A student scores 72 and 84 on the first two tests. The final test is worth twice as much as each of the first two tests. What score must the student get on the final test to achieve an overall weighted average of 85?', 'answer': '92'},]

tasks_df = pd.DataFrame(TASKS)

tasks_df.head()

,id,question,answer
0,Q01,"A jacket costs £80. It is discounted by 25%, a...",66
1,Q02,A student has £150. They spend 35% of it on bo...,73.5
2,Q03,A train travels 125 km in the morning and 18% ...,272.5
3,Q04,A company has 240 employees. 15% work remotely...,153
4,Q05,"A phone costs £600. It is reduced by 20%, then...",432


In [ ]:
tokenizer, model = load_model()
print("Model loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.


### Testing the Tool

In [ ]:
for expression in ["12 * 4 + 3 * 5", "240 / 8", "45 + 20"]:
    print(f"{expression} -> {safe_calculate(expression)}")

12 * 4 + 3 * 5 -> 63
240 / 8 -> 30
45 + 20 -> 65


## 3. Baseline: Direct Reasoning

The model receives each question and is asked to answer directly, without access to a tool. This provides the comparison point for the ReAct condition.

In [ ]:
baseline_preview=[]
for task in TASKS[:5]:
    answer=baseline_answer(tokenizer,model,task["question"])
    baseline_preview.append({"id":task["id"],"expected":task["answer"],"model_answer":answer})
pd.DataFrame(baseline_preview)

,id,expected,model_answer
0,Q01,66,"The jacket is initially discounted by 25%, so ..."
1,Q02,73.5,"First, calculate how much was spent on books:\..."
2,Q03,272.5,"First, let's calculate how far the train trave..."
3,Q04,153,"First, let's find out how many employees work ..."
4,Q05,432,"First, let's calculate the price after the fir..."


## 4. ReAct: Interleaved Reasoning & Action

The loop follows: **Question → Model Decision → Tool Call (if chosen) → Observation → Model Decision → Final Answer**

In [ ]:
example=react_answer(tokenizer,model,TASKS[0]["question"])
print("Final answer:",example["answer"])
print("Tool calls:",example["tool_calls"])
for item in example["trace"]:
    print(f"\nStep {item['step']}")
    print(item["model_output"])
    if "action" in item:
        print("Action:",item["action"])
        print("Observation:",item["observation"])

Final answer: 66
Tool calls: 1

Step 1
{"type": "tool_call", "expression": "80*(1-0.25)+6"}
Action: 80*(1-0.25)+6
Observation: 66

Step 2
{"type": "final", "answer": 66}


In [ ]:
react_preview=[]
for task in TASKS[:5]:
    result=react_answer(tokenizer,model,task["question"])
    react_preview.append({"id":task["id"],"expected":task["answer"],"react_answer":result["answer"],"tool_calls":result["tool_calls"]})
pd.DataFrame(react_preview)

,id,expected,react_answer,tool_calls
0,Q01,66,66.0,1
1,Q02,73.5,174.0,1
2,Q03,272.5,147.5,1
3,Q04,153,153.0,1
4,Q05,432,432.0,1


## 5. Evaluation & Comparison

Both conditions are now run on the full fixed task set. The primary metric is exact numerical correctness after simple answer normalisation; ReAct tool usage is also recorded.

In [ ]:
results=run_experiment(tokenizer,model,TASKS,results_path="week3_results.json")
results_df=pd.DataFrame([{
    "id":r["id"],"expected":r["expected"],"baseline_correct":r["baseline_correct"],
    "react_correct":r["react_correct"],"tool_calls":r["tool_calls"]} for r in results])
results_df

Running Q01...
Running Q02...
Running Q03...
Running Q04...
Running Q05...
Running Q06...
Running Q07...
Running Q08...
Running Q09...
Running Q10...
Running Q11...
Running Q12...
Running Q13...
Running Q14...
Running Q15...
Running Q16...
Running Q17...
Running Q18...
Running Q19...
Running Q20...

Results saved to: week3_results.json


,id,expected,baseline_correct,react_correct,tool_calls
0,Q01,66,True,True,1
1,Q02,73.5,True,False,1
2,Q03,272.5,False,False,1
3,Q04,153,False,True,1
4,Q05,432,False,True,1
5,Q06,94.5,False,True,1
6,Q07,1029,False,False,1
7,Q08,24,True,True,0
8,Q09,480,False,True,1
9,Q10,128,True,True,1


In [ ]:
summary=pd.DataFrame({"Condition":["Direct baseline","ReAct"],"Accuracy":[results_df.baseline_correct.mean(),results_df.react_correct.mean()],"Correct / 20":[results_df.baseline_correct.sum(),results_df.react_correct.sum()]})
summary

,Condition,Accuracy,Correct / 20
0,Direct baseline,0.35,7
1,ReAct,0.55,11


In [ ]:
for result in results[:3]:
    print("="*70)
    print(result["id"],"|",result["question"])
    print("Expected:",result["expected"])
    print("ReAct answer:",result["react_answer"])
    for item in result["trace"]:
        print(f"\nStep {item['step']}:\n{item['model_output']}")
        if "action" in item: print("Action:",item["action"],"| Observation:",item["observation"])

Q01 | A jacket costs £80. It is discounted by 25%, and then a further £6 is added for delivery. What is the final amount paid?
Expected: 66
ReAct answer: 66

Step 1:
{"type": "tool_call", "expression": "80*(1-0.25)+6"}
Action: 80*(1-0.25)+6 | Observation: 66

Step 2:
{"type": "final", "answer": 66}
Q02 | A student has £150. They spend 35% of it on books and then spend another £24. How much money remains?
Expected: 73.5
ReAct answer: 174

Step 1:
{"type": "tool_call", "expression": "(150 * 35 / 100) + 24 + (150 - (150 * 35 / 100))"}
Action: (150 * 35 / 100) + 24 + (150 - (150 * 35 / 100)) | Observation: 174

Step 2:
{"type": "final", "answer": 174}
Q03 | A train travels 125 km in the morning and 18% farther in the afternoon. How many kilometres does it travel altogether?
Expected: 272.5
ReAct answer: 147.5

Step 1:
{"type": "tool_call", "expression": "125 + 125 * 0.18"}
Action: 125 + 125 * 0.18 | Observation: 147.5

Step 2:
{"type": "final", "answer": 147.5}


### Observation

Across the 20 tasks, direct answering gets 7/20 correct (35%), while ReAct gets 11/20 (55%), a gain of four tasks in this small test. The traces show why accuracy alone is not the whole story: some tool expressions encode the wrong operation, and some tasks are answered without a tool; ReAct also loses cases the baseline gets right. These results are suggestive, not conclusive, and should not be generalised beyond this task set.


## 6. Conclusion and Research Takeaway

In this 20-question arithmetic experiment, the ReAct-style condition scored 11/20 (55%), compared with 7/20 (35%) for direct answering. This is a useful improvement on this particular set, but it does not show that ReAct is always better or that the model is reliable. The example traces reveal a key failure mode: a calculator accurately evaluates the expression it receives, even when the model has translated the question into the wrong expression. Tool use can therefore reduce arithmetic errors without fixing errors in task interpretation or operation selection.

For a Research Engineer, the next step is to inspect failures by category, check whether the tool was called appropriately and whether its input represented the question correctly, and evaluate on a larger, varied set with repeatable settings. Report both end-task accuracy and tool-use behaviour, and avoid drawing broad conclusions from 20 examples. The aim is not to demonstrate a perfect LLM, but to understand when tool use helps, where it fails, and what should be measured next.

**Reference:** Yao, S., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. arXiv:2210.03629.
